# Lezione 08 — Modelli di linguaggio n-gramma

**Corso:** NLP Classico con Python · **Blocco 3:** Struttura e modelli del linguaggio
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/08_modelos_lenguaje_ngramas_IT.md`

## Cosa vedrai in questa lezione

- Cos'è un **modello di linguaggio** (la probabilità di una sequenza di parole), la **regola della catena** e l'**ipotesi di Markov**.
- Stimare **bigrammi e trigrammi per conteggio**, prima a mano con un corpus di quattro frasi e poi con due estratti letterari (spagnolo e italiano) e un modello scritto da noi in Python puro.
- **Lisciamento** (Laplace), interpolazione e l'idea di *backoff*, verificando numericamente che le probabilità condizionate sommino a 1.
- **Perplessità** sui dati di test, e perché è confrontabile solo con lo stesso vocabolario, la stessa tokenizzazione e lo stesso trattamento delle parole fuori vocabolario (OOV).
- **Generare** testo campionando il modello e vederne i limiti.

**Requisiti:** lezioni 2 e 4 (tokenizzazione, conteggio di n-grammi). **Librerie:** `numpy` e `matplotlib`; il modello non usa nessuna libreria di modelli di linguaggio.
**Dati:** `datos/gutenberg/quijote_cap01-14.txt` e `datos/gutenberg/promessi_sposi_cap01-06.txt` (in Colab, carica la cartella `datos/`). Sono testi letterari di pubblico dominio di due secoli fa o più: **non rappresentano lo spagnolo né l'italiano di oggi**. Tutte le cifre sono un *esempio illustrativo*.

---
## Cella 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("matplotlib", "matplotlib")
print("Librerie pronte.")

In [ ]:
import os
import re
import math
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

random.seed(42)                          # seme fisso
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

def tabla(cabeceras, filas, ancho=14):
    """Stampa una tabella di testo allineata."""
    print("".join(str(c).ljust(ancho) for c in cabeceras))
    print("-" * ancho * len(cabeceras))
    for fila in filas:
        print("".join((f"{x:.1f}" if isinstance(x, float) else str(x)).ljust(ancho) for x in fila))

print("Configurazione pronta.")

---
## 1. Un modello di linguaggio a mano

Un **modello di linguaggio** assegna una probabilità a una sequenza di parole. La **regola della catena** la scompone esattamente:

$$P(w_1,\dots,w_m)=\prod_{i=1}^{m} P(w_i\mid w_1,\dots,w_{i-1})$$

Poiché non possiamo contare storie intere (quasi nessuna si ripete), l'**ipotesi di Markov** le accorcia: in un bigramma, $P(w_i\mid w_1,\dots,w_{i-1})\approx P(w_i\mid w_{i-1})$, e in un trigramma si usano le due parole precedenti. Si stima per conteggio: $P(w\mid v)=c(v,w)/c(v)$.

Corpus minuscolo (4 frasi). Il codice usa la versione in spagnolo; l'italiana è identica, cambiano solo le parole. Calcola a mano prima di eseguire: `P(gato | un)` (in italiano `P(gatto | il)`), e la probabilità della frase `un gato come pescado` (in italiano `il gatto mangia pesce`).

In [ ]:
# datos-es
# Corpus minuscolo (4 frasi per lingua) da calcolare a mano
CHICO = {
    "spa": [["un", "gato", "come", "pescado"], ["un", "gato", "duerme"],
            ["un", "perro", "come", "carne"], ["un", "perro", "duerme"]],
    "ita": [["il", "gatto", "mangia", "pesce"], ["il", "gatto", "dorme"],
            ["il", "cane", "mangia", "carne"], ["il", "cane", "dorme"]],
}

In [ ]:
# Token speciali del modello
INI, FIN, UNK = "<s>", "</s>", "<unk>"     # inizio di frase, fine di frase, parola fuori vocabolario (OOV)
ENCAB = re.compile(r"^(Capítulo|CAPITOLO)\b")   # dato-es

def capitulos(ruta):
    """Legge un estratto e restituisce una lista di capitoli; ogni capitolo è una lista di paragrafi (senza l'intestazione)."""
    caps = []
    for parrafo in open(ruta, encoding="utf-8").read().split("\n\n"):
        parrafo = parrafo.strip()
        if not parrafo:
            continue
        if ENCAB.match(parrafo):
            caps.append([])
        else:
            caps[-1].append(parrafo)
    return caps

def oraciones(parrafos, puntuacion=False):
    """Divide in frasi (. ! ? …) e in token minuscoli; per default i segni di punteggiatura non sono token."""
    patron = r"\w+'|\w+|[,;:.!?¿¡«»—]" if puntuacion else r"\w+'|\w+"   # dato-es
    salida = []
    for p in parrafos:
        for o in re.split(r"(?<=[.!?…])\s+", p):
            toks = re.findall(patron, o.lower())
            if toks:
                salida.append(toks)
    return salida

def contar(oras, n):
    """Conta n-grammi con n-1 marche <s> all'inizio e una </s> alla fine. Restituisce (n-grammi, contesti)."""
    c, ctx = Counter(), Counter()
    for o in oras:
        seq = [INI] * (n - 1) + o + [FIN]
        for i in range(n - 1, len(seq)):
            c[tuple(seq[i - n + 1:i + 1])] += 1
            ctx[tuple(seq[i - n + 1:i])] += 1
    return c, ctx

def vocabulario(oras, minimo):
    """Parole con almeno `minimo` occorrenze nell'addestramento."""
    cuenta = Counter(t for o in oras for t in o)
    return {t for t, k in cuenta.items() if k >= minimo}

def aplicar_vocab(oras, vocab):
    """Sostituisce con <unk> ciò che non è nel vocabolario."""
    return [[t if t in vocab else UNK for t in o] for o in oras]

In [ ]:
# Calcolo a mano con il corpus minuscolo (spagnolo; l'italiano è identico salvo le parole)
idioma = "spa"
oras = CHICO[idioma]
bi, ctx1 = contar(oras, 2)
print("Bigrammi contati:")
for par, k in sorted(bi.items()):
    print("  ", par, k, "/", ctx1[par[:1]], "=", k / ctx1[par[:1]])

def p_bigrama(w, v):
    return bi[(v, w)] / ctx1[(v,)]

frase = oras[0] + [FIN]
p = p_bigrama(frase[0], INI)
for v, w in zip(frase, frase[1:]):
    p *= p_bigrama(w, v)
print("\nP(gato | un) =", p_bigrama("gato", "un"))
print("P(", " ".join(oras[0]), ") =", p)
assert p_bigrama("gato", "un") == 0.5 and p == 0.125             # coincide con il calcolo a mano: 1 · 1/2 · 1/2 · 1/2 · 1

### Cosa osservi?

`P(gato | un) = 2/4`: `un` compare 4 volte a inizio di frase (`<s> un`) e 2 di queste sono seguite da `gato` (in italiano: `il` e `gatto`). La frase completa vale `1 · 0,5 · 0,5 · 0,5 · 1 = 0,125`; l'ultimo fattore `P(</s> | pescado)` è 1 perché `pescado` chiude sempre la frase. Senza la marca `<s>` non avremmo la probabilità di **iniziare** con `un`, e senza `</s>` un modello non saprebbe **quando finire**: per questo le aggiungiamo. (`<s>` e `</s>` sono token speciali: segnano inizio e fine di frase; il terzo, `<unk>`, lo introduciamo nella sezione 2.)

---
## 2. Dati reali, token speciali e OOV

Definiamo ciò che userà tutto il modello (e che **deve essere identico** per addestrare e per valutare):

- **Tokenizzazione:** minuscole; un token è una parola (`\w+`), e in italiano un articolo o una preposizione elisa con apostrofo (`dell'`, `l'`) è un token a parte. I segni di punteggiatura **non** sono token (più avanti vedremo come cambia la perplessità se lo fossero). Le frasi si separano dopo `.`, `!`, `?` o `…`.
- `<s>`: marca di inizio frase (si antepone n−1 volte); `</s>`: fine frase, che il modello **predice**; `<unk>`: sostituisce qualsiasi parola **fuori vocabolario (OOV)**.
- **Trattamento dell'OOV:** il vocabolario sono le parole con almeno 2 occorrenze nell'**addestramento**; il resto (in addestramento e in test) passa a `<unk>`. Il vocabolario si fissa con l'addestramento e **non si tocca** con i dati di test.
- **Divisione per capitoli contigui, non a caso:** addestramento = primi capitoli, sviluppo = i successivi (per scegliere gli iperparametri) e test = gli ultimi. È ciò che accade nella pratica: il modello si usa su testo che arriva **dopo**.

**Avvertenza:** un corpus letterario di due secoli fa non è lo spagnolo né l'italiano di oggi (`lantejas`, `desta`, `mesmo`, `talchè`): ciò che il modello impara è la lingua di quel libro.

In [ ]:
CORPORA = {"spa": "gutenberg/quijote_cap01-14.txt", "ita": "gutenberg/promessi_sposi_cap01-06.txt"}
# Capitoli contigui (indici da 0): addestramento, sviluppo, test
DIVISION = {"spa": (range(0, 10), range(10, 12), range(12, 14)),
            "ita": (range(0, 4), range(4, 5), range(5, 6))}

CAPS, TRAIN, DEV, TEST = {}, {}, {}, {}
for idioma, ruta in CORPORA.items():
    CAPS[idioma] = capitulos(cargar_datos(ruta))
    tr, dv, te = DIVISION[idioma]
    TRAIN[idioma], DEV[idioma], TEST[idioma] = ([o for i in r for o in oraciones(CAPS[idioma][i])] for r in (tr, dv, te))
    palabras = sum(len(p.split()) for cap in CAPS[idioma] for p in cap)
    print(idioma, "capitoli:", len(CAPS[idioma]), "| parole (circa):", palabras)

tabla(["lingua", "partizione", "frasi", "tokens"],
      [[i, nombre, len(d[i]), sum(map(len, d[i]))] for i in CORPORA for nombre, d in (("train", TRAIN), ("dev", DEV), ("test", TEST))])
assert all(sum(map(len, TRAIN[i])) > 20000 for i in CORPORA)

### Tre cifre da guardare sempre

La dimensione di ogni partizione e, nella sezione successiva, il **tasso di OOV** del test. Con solo circa 21 000 parole di addestramento per lingua, molte parole del test saranno sconosciute: è un corpus piccolo (*esempio illustrativo*) e i modelli veri si addestrano con molto più testo.

---
## 3. Il modello completo e il problema degli zeri

Scriviamo il modello (conteggi con `Counter`, probabilità con funzioni): stima di massima verosimiglianza (`p_mle`), lisciamento di Laplace (`p_laplace`), interpolazione lineare di trigramma, bigramma e unigramma (`p_interp`) e un *backoff* semplice (`p_backoff`) che useremo solo per vedere che non è una distribuzione di probabilità.

In [ ]:
class ModeloLM:
    """Modello di linguaggio n-gramma in Python puro (Counter e dict)."""
    def __init__(self, oras_train, n=3, minimo=2):
        self.n = n
        self.vocab0 = vocabulario(oras_train, minimo)          # parole note
        oras = aplicar_vocab(oras_train, self.vocab0)
        self.V = sorted(set(t for o in oras for t in o) | {FIN, UNK})   # ciò che si può predire (senza <s>)
        self.c, self.ctx = {}, {}
        for k in range(1, n + 1):                               # un insieme di conteggi per ordine
            self.c[k], self.ctx[k] = contar(oras, k)

    def mapear(self, oras):
        """Applica al testo di test lo STESSO vocabolario dell'addestramento."""
        return aplicar_vocab(oras, self.vocab0)

    def _h(self, h, k):
        return tuple(h[len(h) - (k - 1):]) if k > 1 else ()

    def p_mle(self, w, h, k):
        """Massima verosimiglianza: c(h, w) / c(h). Vale 0 se l'n-gramma non è mai comparso."""
        hh = self._h(h, k)
        d = self.ctx[k][hh]
        return self.c[k][hh + (w,)] / d if d else 0.0

    def p_laplace(self, w, h, k, a=1.0):
        """Lisciamento di Laplace (add-a): (c(h, w) + a) / (c(h) + a·|V|)."""
        hh = self._h(h, k)
        return (self.c[k][hh + (w,)] + a) / (self.ctx[k][hh] + a * len(self.V))

    def p_interp(self, w, h, lam):
        """Interpolazione lineare; lam = (peso dell'ordine n, ..., peso dell'unigramma). Un ordine il cui contesto non esiste cede il suo peso agli altri."""
        num = den = 0.0
        for peso, k in zip(lam, range(self.n, 0, -1)):
            hh = self._h(h, k)
            if self.ctx[k][hh] > 0 and peso > 0:
                num += peso * self.c[k][hh + (w,)] / self.ctx[k][hh]
                den += peso
        return num / den

    def p_backoff(self, w, h, k, alfa=0.4):
        """Stupid backoff: NON è una distribuzione di probabilità (non somma 1)."""
        hh = self._h(h, k)
        if self.c[k][hh + (w,)] > 0:
            return self.c[k][hh + (w,)] / self.ctx[k][hh]
        return alfa * self.p_backoff(w, h, k - 1, alfa) if k > 1 else 0.0

def perplejidad(modelo, oras, prob):
    """exp(-media di log P) sui token di `oras` e i loro </s>; non conta <s>. prob(w, storia) -> probabilità."""
    logsum, N = 0.0, 0
    for o in modelo.mapear(oras):
        seq = [INI] * (modelo.n - 1) + o + [FIN]
        for i in range(modelo.n - 1, len(seq)):
            p = prob(seq[i], seq[:i])
            if p <= 0:
                return float("inf")                  # un solo zero rende infinita la perplessità
            logsum += math.log(p)
            N += 1
    return math.exp(-logsum / N)

def generar(modelo, lam, rng, max_tokens=30):
    """Campiona parola per parola dal modello interpolato fino a </s> o max_tokens."""
    hist = [INI] * (modelo.n - 1)
    salida = []
    while len(salida) < max_tokens:
        pesos = [modelo.p_interp(w, hist, lam) for w in modelo.V]
        w = rng.choices(modelo.V, weights=pesos, k=1)[0]
        if w == FIN:
            break
        salida.append(w)
        hist.append(w)
    return salida

In [ ]:
MODELOS = {i: ModeloLM(TRAIN[i], n=3, minimo=2) for i in CORPORA}
for i, m in MODELOS.items():
    toks_test = [t for o in TEST[i] for t in o]
    oov = sum(t not in m.vocab0 for t in toks_test) / len(toks_test)
    # n-grammi del test già visti nell'addestramento (con il vocabolario dell'addestramento)
    vistos = {}
    for k in (2, 3):
        cnt = [tuple(([INI] * (k - 1) + o + [FIN])[j - k + 1:j + 1]) for o in m.mapear(TEST[i]) for j in range(k - 1, len(o) + k)]
        vistos[k] = sum(x in m.c[k] for x in cnt) / len(cnt)
    print(i, "|V| =", len(m.V), "| OOV nel test:", f"{100 * oov:.1f}%",
          "| bigrammi visti:", f"{100 * vistos[2]:.1f}%", "| trigrammi visti:", f"{100 * vistos[3]:.1f}%")
    print("   perplessità sul test con MLE:",
          {k: perplejidad(m, TEST[i], lambda w, h, k=k: m.p_mle(w, h, k)) for k in (1, 2, 3)})

### Cosa osservi?

Con la massima verosimiglianza, la perplessità del bigramma e del trigramma è **infinita**: basta **un** bigramma del test che non si è visto nell'addestramento perché una probabilità sia 0 e tutto il prodotto si annulli. Una percentuale alta dei trigrammi del test non compare mai nell'addestramento: contare non basta, bisogna **spartire un po' di probabilità a ciò che non si è visto**. Questo è il **lisciamento**. Solo l'unigramma (non dipende dalla storia) risulta finito.

---
## 4. Verifica: le probabilità condizionate sommano a 1

Un modello ben definito soddisfa $\sum_{w\in V} P(w\mid h)=1$ per **ogni** storia $h$. Lo verifichiamo per tre storie: quella di inizio frase, la più frequente nell'addestramento e una mai vista. (`V` include `</s>` e `<unk>` e **non** include `<s>`, che non si predice mai.)

In [ ]:
def suma(m, prob, h):
    return sum(prob(w, h) for w in m.V)

LAM0 = (0.3, 0.4, 0.3)                                  # pesi di esempio: trigramma, bigramma, unigramma
for i, m in MODELOS.items():
    frecuente = max((k for k in m.ctx[2] if UNK not in k and INI not in k), key=m.ctx[2].get)
    historias = {"inizio": [INI, INI], "più frequente": [INI] + list(frecuente), "mai vista": ["zzz", "qqq"]}
    filas = []
    for nombre, h in historias.items():
        filas.append([nombre, round(suma(m, lambda w, h_: m.p_mle(w, h_, 3), h), 6),
                      round(suma(m, lambda w, h_: m.p_laplace(w, h_, 3), h), 6),
                      round(suma(m, lambda w, h_: m.p_interp(w, h_, LAM0), h), 6),
                      round(suma(m, lambda w, h_: m.p_backoff(w, h_, 3), h), 3)])
        assert abs(filas[-1][2] - 1) < 1e-9 and abs(filas[-1][3] - 1) < 1e-9      # Laplace e interpolazione sommano a 1
        assert filas[-1][1] in (0.0, 1.0) or abs(filas[-1][1] - 1) < 1e-9
    print(i, "storia più frequente:", frecuente)
    tabla(["storia", "MLE", "Laplace", "interpolazione", "backoff"], filas, 16)

### Cosa osservi?

- **Laplace** e **interpolazione** sommano a 1 in tutte e tre le storie (inclusa quella mai vista: l'interpolazione cede il peso del trigramma e del bigramma all'unigramma, che esiste sempre).
- L'**MLE** somma a 1 solo se la storia è stata vista; con una storia sconosciuta somma 0: non è definito.
- Il *backoff* semplice (`alfa = 0,4`) **non somma 1** (≈ 1,1 e 1,2-1,3 nelle storie viste, ≈ 0,2 in quella sconosciuta): dà punteggi utili per ordinare i candidati, ma **non è una distribuzione**; per questo non ne calcoliamo la perplessità. Con sconti e normalizzazione (Katz, Kneser-Ney) lo sarebbe; qui teniamo solo l'idea di «ripiegare su un n-gramma più corto».

---
## 5. Perplessità: la definizione e un calcolo a mano

La **perplessità** di un testo di $N$ token è $\mathrm{PP}=\exp\!\big(-\tfrac1N\sum_i \ln P(w_i\mid h_i)\big)$: l'inverso della probabilità media per token (media geometrica). Si può leggere come «tra quante parole equiprobabili esita il modello a ogni passo». Qui $N$ conta i token e i `</s>`, non i `<s>`.

Con il corpus minuscolo e un bigramma MLE, la frase di test `un perro duerme` (in italiano `il cane dorme`) ha $P=1\cdot\frac12\cdot\frac12\cdot1=\frac14$ con $N=4$ (`un`, `perro`, `duerme`, `</s>`): $\mathrm{PP}=(1/4)^{-1/4}=\sqrt2\approx1{,}414$. Un modello uniforme su $|V|$ parole avrebbe $\mathrm{PP}=|V|$: serve da riferimento.

In [ ]:
chico = ModeloLM(CHICO["spa"], n=2, minimo=1)
prueba = [["un", "perro", "duerme"]]                     # dato-es
pp = perplejidad(chico, prueba, lambda w, h: chico.p_mle(w, h, 2))
pp_uniforme = perplejidad(chico, prueba, lambda w, h: 1 / len(chico.V))
print("perplessità a mano:", math.sqrt(2), "| codice:", pp)
print("|V| =", len(chico.V), "| perplessità del modello uniforme:", pp_uniforme)
assert abs(pp - math.sqrt(2)) < 1e-12 and abs(pp_uniforme - len(chico.V)) < 1e-9

# Laplace sullo stesso corpus: P(pescado | un) = 1 / (4 + |V|) e la riga somma 1
p_l = chico.p_laplace("pescado", [INI, "un"], 2)
print("P_Laplace(pescado | un) = 1/", 4 + len(chico.V), "=", p_l)
assert abs(p_l - 1 / (4 + len(chico.V))) < 1e-12
assert abs(sum(chico.p_laplace(w, [INI, "un"], 2) for w in chico.V) - 1) < 1e-12

### Cosa osservi?

La perplessità calcolata a mano e quella del codice coincidono (√2). Con `|V| = 9` (le 8 parole viste, incluso `</s>`, più `<unk>`), il modello uniforme avrebbe perplessità 9. Il lisciamento di Laplace (`+1` a ogni n-gramma) trasforma `P(pescado | un) = 0` in `1/13`, ma lo **toglie** agli n-grammi visti: `P(gato | un)` scende da 1/2 a 3/13.

---
## 6. Esperimento guidato 1 — ordine del modello, lisciamento e perplessità sul test

Confrontiamo, con lo stesso vocabolario e la stessa tokenizzazione, unigramma, bigramma e trigramma con Laplace (`a = 1`), con Laplace `a = 0,01` e con interpolazione. I pesi dell'interpolazione **non** si scelgono guardando il test: si cercano in una griglia sull'insieme di **sviluppo**.

In [ ]:
# Griglia di pesi (trigramma, bigramma, unigramma) che sommano a 1, con peso dell'unigramma ≥ 0,1
REJILLA = [(a / 10, b / 10, (10 - a - b) / 10) for a in range(0, 9) for b in range(0, 10 - a) if 10 - a - b >= 1]

MEJOR = {}
for i, m in MODELOS.items():
    puntos = [(perplejidad(m, DEV[i], lambda w, h, lam=lam: m.p_interp(w, h, lam)), lam) for lam in REJILLA]
    MEJOR[i] = min(puntos)[1]
    print(i, "migliori pesi in sviluppo:", MEJOR[i], "| PP dev =", round(min(puntos)[0], 1))

filas = []
for i, m in MODELOS.items():
    fila = [i]
    for k in (1, 2, 3):
        fila.append(round(perplejidad(m, TEST[i], lambda w, h, k=k: m.p_laplace(w, h, k, 1.0)), 1))
    fila.append(round(perplejidad(m, TEST[i], lambda w, h: m.p_laplace(w, h, 3, 0.01)), 1))
    fila.append(round(perplejidad(m, TEST[i], lambda w, h: m.p_interp(w, h, MEJOR[i])), 1))
    fila.append(round(perplejidad(m, TRAIN[i], lambda w, h: m.p_mle(w, h, 3)), 1))
    filas.append(fila)
tabla(["lingua", "1-g Laplace", "2-g Laplace", "3-g Laplace", "3-g a=0,01", "interpolato", "3-g MLE train"], filas, 15)
assert all(f[5] < min(f[1], f[2], f[3]) for f in filas)          # l'interpolazione vince sul test

### Cosa osservi?

Con questi dati (*esempio illustrativo*, un corpus piccolo):

- Con Laplace `a = 1` il trigramma è **peggiore** del bigramma e questo peggiore dell'unigramma: con |V| ≈ 1 600 e pochi dati, «+1» distribuisce troppa probabilità a ciò che non si è visto (il lisciamento di Laplace è troppo aggressivo per n-grammi di ordine alto). Con `a = 0,01` migliora, ma resta indietro.
- L'**interpolazione** (pesi scelti in sviluppo, con molto peso agli ordini bassi) vince sul test in entrambe le lingue.
- La perplessità del trigramma MLE **sull'addestramento** è molto bassa (il modello memorizza), mentre sul test era infinita: misurare sempre su dati tenuti da parte.
- Non guardare le cifre di spagnolo e italiano come una gara: nella sezione successiva vediamo perché non si possono confrontare.

---
## 7. Esperimento guidato 2 — la perplessità è confrontabile solo con lo stesso vocabolario

Sullo **stesso testo di test e lo stesso modello**, cambiamo solo la soglia di OOV (una parola entra nel vocabolario se compare almeno `minimo` volte) e, a parte, la tokenizzazione (con i segni di punteggiatura come token). I pesi dell'interpolazione sono quelli di sviluppo. Prima di eseguire, prevedi: la perplessità dovrebbe cambiare?

In [ ]:
UMBRALES = [1, 2, 3, 5, 10, 20, 50, 10 ** 6]
RES_UMBRAL = {}
for i in CORPORA:
    filas = []
    for minimo in UMBRALES:
        m = ModeloLM(TRAIN[i], n=3, minimo=minimo)
        toks = [t for o in TEST[i] for t in o]
        oov = 100 * sum(t not in m.vocab0 for t in toks) / len(toks)
        pp = perplejidad(m, TEST[i], lambda w, h: m.p_interp(w, h, MEJOR[i]))
        filas.append([minimo if minimo < 10 ** 6 else "tutto <unk>", len(m.V), round(oov, 1), round(pp, 1)])
    RES_UMBRAL[i] = filas
    print(i, "(stesso testo di test, stessi pesi; cambia solo il vocabolario)")
    tabla(["minimo", "|V|", "OOV test %", "perplessità"], filas, 16)

# Tokenizzazione con punteggiatura: un altro «vocabolario» sullo stesso testo
for i in CORPORA:
    tr, dv, te = DIVISION[i]
    T = [o for k in tr for o in oraciones(CAPS[i][k], puntuacion=True)]
    P = [o for k in te for o in oraciones(CAPS[i][k], puntuacion=True)]
    m = ModeloLM(T, n=3, minimo=2)
    print(i, "con la punteggiatura come token: |V| =", len(m.V),
          "| perplessità =", round(perplejidad(m, P, lambda w, h: m.p_interp(w, h, MEJOR[i])), 1))

fig, ax = plt.subplots()
for i, filas in RES_UMBRAL.items():
    ax.plot([f[2] for f in filas[1:-1]], [f[3] for f in filas[1:-1]], marker="o", label=i)
ax.set_xlabel("% di OOV nel test"); ax.set_ylabel("perplessità"); ax.set_yscale("log")
ax.set_title("Più OOV, «migliore» perplessità: un'illusione"); ax.legend()
plt.show()
assert RES_UMBRAL["spa"][-1][3] < 2                       # tutto <unk>: perplessità quasi 1
assert RES_UMBRAL["spa"][0][3] == float("inf")            # minimo=1: <unk> mai visto in addestramento

### Cosa osservi?

- Con lo **stesso testo di test**, alzare la soglia di OOV fa **scendere la perplessità** (in spagnolo da 78,0 a 14,1 con `minimo = 20`; in italiano da 102,6 a 17,4) e fino a ~1 se tutto diventa `<unk>`. Il modello non è migliorato: ha **cambiato la domanda** (bisogna predire sempre meno parole distinte, e `<unk>` è molto facile da predire). Un modello che trasforma tutto in `<unk>` avrebbe perplessità quasi 1 e non saprebbe nulla della lingua.
- Con `minimo = 1` non esiste alcun `<unk>` nell'addestramento e qualsiasi parola nuova del test ha probabilità 0: perplessità infinita. Trattare l'OOV è **obbligatorio**, e va dichiarato.
- Aggiungere la punteggiatura come token abbassa anch'essa la perplessità (i segni sono facili da predire e contano come token): di nuovo, un altro vocabolario.
- **Conclusione:** confrontare perplessità ha senso solo con lo stesso vocabolario, la stessa tokenizzazione e lo stesso trattamento dell'OOV (e, meglio ancora, lo stesso testo di test). **Confrontare lo spagnolo con l'italiano non ha senso**: cambiano il testo, il vocabolario (|V|), il tasso di OOV, la morfologia, la tokenizzazione (gli apostrofi italiani) e l'autore e il genere. Le cifre delle due lingue di questa lezione si leggono separatamente.

---
## 8. Esperimento guidato 3 — capitoli contigui contro frasi a caso

Ripetiamo la valutazione con una divisione **a caso frase per frase** (5 semi), con le stesse dimensioni di addestramento e test e le stesse regole di OOV, e confrontiamo con la divisione per capitoli. Guardiamo tre indicatori: il tasso di OOV nel test, la copertura di bigrammi e trigrammi del test, e la perplessità.

In [ ]:
def indicadores(train, test, lam):
    m = ModeloLM(train, n=3, minimo=2)
    toks = [t for o in test for t in o]
    oov = 100 * sum(t not in m.vocab0 for t in toks) / len(toks)
    cob = []
    for k in (2, 3):
        cnt = [tuple(([INI] * (k - 1) + o + [FIN])[j - k + 1:j + 1]) for o in m.mapear(test) for j in range(k - 1, len(o) + k)]
        cob.append(100 * sum(x in m.c[k] for x in cnt) / len(cnt))
    return [round(oov, 1), round(cob[0], 1), round(cob[1], 1), round(perplejidad(m, test, lambda w, h: m.p_interp(w, h, lam)), 1)]

def reparto_azar(todas, n_train, n_test, semilla):
    """Mescola le frasi e divide fino a raggiungere le dimensioni (in token) della divisione per capitoli."""
    rng = random.Random(semilla)
    o = todas[:]
    rng.shuffle(o)
    prueba, resto, c = [], [], 0
    for s in o:
        if c < n_test:
            prueba.append(s); c += len(s)
        else:
            resto.append(s)
    train, c = [], 0
    for s in resto:
        if c < n_train:
            train.append(s); c += len(s)
    return train, prueba

for i in CORPORA:
    todas = [o for cap in CAPS[i] for o in oraciones(cap)]
    n_train, n_test = sum(map(len, TRAIN[i])), sum(map(len, TEST[i]))
    filas = [["capitoli", ""] + indicadores(TRAIN[i], TEST[i], MEJOR[i])]
    for semilla in range(5):
        tr, te = reparto_azar(todas, n_train, n_test, semilla)
        filas.append(["a caso", semilla] + indicadores(tr, te, MEJOR[i]))
    print(i)
    tabla(["divisione", "seme", "OOV %", "bigr. visti %", "trigr. visti %", "perplessità"], filas, 16)

### Cosa osservi?

- In **spagnolo**, la divisione a caso riduce il tasso di OOV del test (da 22,7 % a 16,6-18,4 %): il test condivide con l'addestramento personaggi, luoghi e passaggi degli stessi capitoli. È una **fuga di informazione** (*leakage*) dal futuro verso l'addestramento.
- Ma le altre cifre **non danno un segnale pulito**: la copertura di bigrammi e trigrammi non sale e la perplessità sale (83-93 contro 78,0). Con meno OOV ci sono meno `<unk>` (economici da predire), e inoltre il test per capitoli e il test a caso sono testi diversi. La variazione tra semi (da 83 a 93) è dell'ordine della differenza con la divisione per capitoli: **due perplessità con diverso tasso di OOV o diverso testo di test non sono confrontabili** (stesso avviso della sezione 7).
- In **italiano**, in questo estratto, quasi non ci sono differenze. Che non si vedano non prova che non ci sia fuga di informazione: un estratto piccolo e omogeneo non la rileva.
- **Regola pratica:** la divisione non si sceglie per la cifra che viene meglio, ma per ciò che imita l'uso reale: il modello verrà applicato a testo che arriva dopo, quindi si divide in blocchi contigui (capitoli, mesi…) e lo si dichiara. Un corpus letterario antico non è nemmeno rappresentativo dello spagnolo o dell'italiano di oggi: le cifre valgono per quel libro.

---
## 9. Lingua di due secoli fa contro lingua di oggi

Proviamo ogni modello su recensioni **moderne** del corso (`texto_limpio` di `resenas_es.csv` e `resenas_it.csv`, sintetiche): lo stesso vocabolario e la stessa tokenizzazione, un altro dominio e un'altra epoca.

In [ ]:
import csv
for i, archivo in (("spa", "resenas_es.csv"), ("ita", "resenas_it.csv")):
    with open(cargar_datos(archivo), encoding="utf-8", newline="") as f:
        resenas = [fila["texto_limpio"] for fila in csv.DictReader(f)]
    modernas = oraciones(resenas)
    m = MODELOS[i]
    toks = [t for o in modernas for t in o]
    oov_mod = 100 * sum(t not in m.vocab0 for t in toks) / len(toks)
    toks_te = [t for o in TEST[i] for t in o]
    oov_te = 100 * sum(t not in m.vocab0 for t in toks_te) / len(toks_te)
    print(i, "OOV nelle recensioni moderne:", f"{oov_mod:.1f}%", "| OOV nel test letterario:", f"{oov_te:.1f}%")
    print("   parole di oggi sconosciute al modello (le 8 più frequenti):", [w for w, _ in Counter(t for t in toks if t not in m.vocab0).most_common(8)])
    assert oov_mod > oov_te

### Cosa osservi?

Il tasso di OOV su testo moderno è maggiore (≈ 38 % contro ≈ 22,7 % in spagnolo; ≈ 40 % contro ≈ 20,9 % in italiano) che sul test letterario della stessa opera: il modello addestrato con un libro classico conosce poco del vocabolario di oggi (guarda le parole sconosciute più frequenti che vengono stampate). Non lo useremmo per nulla di attuale. E questa cifra non è una perplessità confrontabile con le precedenti.

---
## 10. Generare testo per campionamento e i suoi limiti

Per generare, partiamo da `<s> <s>` e a ogni passo **campioniamo** la parola successiva secondo $P(w\mid h)$ fino a ottenere `</s>`. Con il seme fisso il risultato è riproducibile. Confrontiamo tre modelli (unigramma, bigramma e trigramma, tramite i pesi dell'interpolazione) e misuriamo quanta parte del testo generato sono trigrammi copiati dall'addestramento.

In [ ]:
ORDENES = {"unigramma": (0.0, 0.0, 1.0), "bigramma": (0.0, 0.9, 0.1), "trigramma": (0.8, 0.15, 0.05)}
for i, m in MODELOS.items():
    c3 = contar(m.mapear(TRAIN[i]), 3)[0]
    filas = []
    print(i)
    for nombre, lam in ORDENES.items():
        rng = random.Random(2026)                          # seme fisso
        muestras = [generar(m, lam, rng, 30) for _ in range(30)]
        tri = [tuple(([INI, INI] + g)[j - 2:j + 1]) for g in muestras for j in range(2, len(g) + 2)]
        copiados = 100 * sum(x in c3 for x in tri) / len(tri)
        unk = 100 * sum(g.count(UNK) for g in muestras) / sum(map(len, muestras))
        filas.append([nombre, round(copiados, 1), round(unk, 1), round(sum(map(len, muestras)) / len(muestras), 1)])
        print("  ", nombre, ":", " ".join(generar(m, lam, random.Random(7), 25)))
    tabla(["modello", "trigr. in train %", "<unk> %", "lung. media"], filas, 22)
    assert filas[2][1] > filas[1][1] > filas[0][1]

### Cosa osservi?

- L'**unigramma** produce una «insalata di parole»; il **bigramma** produce coppie plausibili ma senza senso globale; il **trigramma** suona più fluido perché **copia** frammenti dell'addestramento (più dell'80 % dei suoi trigrammi erano già nel testo): con così poco testo, un trigramma quasi memorizza.
- Compaiono `<unk>` nel testo: il modello non sa quale parola ci sia dietro.
- Nulla garantisce **coerenza a lunga distanza** (concordanza, tema, trama): il contesto è di sole una o due parole. Inoltre, con il seme fisso tutto è riproducibile, ma **un altro seme dà un altro testo**: non esiste un testo «giusto».
- I modelli di linguaggio moderni (reti neurali, LLM) seguono la stessa idea (predire la parola successiva e campionare), ma con contesti molto più lunghi.

---
## Riepilogo

| Concetto | Idea chiave |
|---|---|
| Regola della catena e Markov | la probabilità della frase è un prodotto di condizionate, accorciate alle n−1 parole precedenti |
| Conteggio e token speciali | `<s>`, `</s>` e `<unk>` si dichiarano; P = c(h, w) / c(h) |
| Lisciamento | l'MLE dà 0 a ciò che non si è visto; Laplace è aggressivo; l'interpolazione combina gli ordini; il *backoff* semplice non è una distribuzione |
| Verifica | le condizionate sommano a 1 per ogni storia |
| Perplessità | confrontabile solo con lo stesso vocabolario, tokenizzazione e trattamento dell'OOV; spagnolo contro italiano non si confrontano |
| Divisione | per blocchi contigui, non a caso; corpus antico ≠ lingua di oggi |
| Generazione | campionare riproduce localmente il corpus, senza coerenza globale |

**Prossima:** nella lezione 9 passiamo dalle sequenze ai **temi** di un corpus (*topic modeling*: LSA e LDA). Poi la pratica di questa lezione.